# 🔥 Exp A: Deep Learning (`PyTorch` Attention-BiLSTM & Residual N-BEATS) vs. Classical Econometrics (`SARIMAX` & `LightGBM`) for Multi-Horizon Forecasting
**Author:** Akmal Khan, PhD  
**Repository:** [github.com/akmalkhaniub/enterprise-banking-ds-platform](https://github.com/akmalkhaniub/enterprise-banking-ds-platform)

### 🎯 Research Question
When does GPU-accelerated Deep Learning (`Temporal Attention Bi-LSTM` and `Residual N-BEATS` blocks) outperform classical statistical econometrics (`statsmodels` `SARIMAX`) and gradient boosted trees (`LightGBM` / `GradientBoostingRegressor`) on multi-seasonal transactional time series?

In [ ]:
import time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import GradientBoostingRegressor
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Active Compute Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"})')
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
# 1. Generate 730 Days (2 Years) of Multi-Seasonal Non-Linear Demand Series (Weekly + Monthly Payday + Macro Regime Shifts)
n_days = 730
dates = pd.date_range('2025-01-01', periods=n_days, freq='D')
t = np.arange(n_days, dtype=float)
dow = dates.dayofweek.values
dom = dates.day.values

is_payday = ((dom <= 6) | (dom >= 28)).astype(float)
is_weekend = (dow >= 5).astype(float)
macro_rate = 17.5 - 0.006 * t + 0.4 * np.sin(2 * np.pi * t / 180.0)

# Non-linear interaction between payday window, weekly cycle, and autoregressive shock
weekly_wave = 1.0 + 0.22 * np.sin(2 * np.pi * dow / 7.0) + 0.15 * is_weekend
payday_boost = 1.0 + 0.36 * is_payday + 0.08 * (is_payday * is_weekend)
trend = 320.0 + 0.18 * t + 18.0 * np.sin(2 * np.pi * t / 365.25)
noise = np.random.normal(0, 8.5, size=n_days)

y = trend * weekly_wave * payday_boost - 2.4 * macro_rate + noise
df = pd.DataFrame({
    'date': dates,
    'demand_millions': y,
    'is_payday': is_payday,
    'is_weekend': is_weekend,
    'macro_rate': macro_rate,
    'dow': dow,
    'dom': dom
})
print(f'Generated {len(df)} daily observations | Mean: {y.mean():.2f}M | Std: {y.std():.2f}M')

In [ ]:
# 2. Classical Baselines: SARIMAX(1,0,1)(1,0,1,7) & Lag-Feature Gradient Boosting
horizon = 60
train_df = df.iloc[:-horizon].copy()
test_df = df.iloc[-horizon:].copy()
exog_cols = ['is_payday', 'is_weekend', 'macro_rate']

t0 = time.perf_counter()
sarimax = SARIMAX(
    train_df['demand_millions'].values,
    exog=train_df[exog_cols].values,
    order=(1, 0, 1),
    seasonal_order=(1, 0, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False, maxiter=100)
sarimax_preds = np.asarray(sarimax.get_forecast(steps=horizon, exog=test_df[exog_cols].values).predicted_mean)
sarimax_time = time.perf_counter() - t0

# Feature engineering for Tree Model
df_feat = df.copy()
for lag in [1, 2, 7, 14, 28]:
    df_feat[f'lag_{lag}'] = df_feat['demand_millions'].shift(lag)
df_feat['roll_mean_7'] = df_feat['demand_millions'].shift(1).rolling(7).mean()
df_feat['roll_std_7'] = df_feat['demand_millions'].shift(1).rolling(7).std()
df_feat = df_feat.dropna().reset_index(drop=True)

feat_cols = ['is_payday', 'is_weekend', 'macro_rate', 'dow', 'dom', 'lag_1', 'lag_2', 'lag_7', 'lag_14', 'lag_28', 'roll_mean_7', 'roll_std_7']
t0 = time.perf_counter()
gbr = GradientBoostingRegressor(n_estimators=250, learning_rate=0.04, max_depth=4, random_state=42)
gbr.fit(df_feat.iloc[:-horizon][feat_cols], df_feat.iloc[:-horizon]['demand_millions'])
gbr_preds = gbr.predict(df_feat.iloc[-horizon:][feat_cols])
gbr_time = time.perf_counter() - t0
print(f'SARIMAX trained in {sarimax_time:.2f}s | GBR trained in {gbr_time:.2f}s')

In [ ]:
# 3. Deep Learning Models in PyTorch: (A) Temporal Attention Bi-LSTM & (B) Residual N-BEATS Style Block
seq_len = 28
raw_features = df[['demand_millions', 'is_payday', 'is_weekend', 'macro_rate']].values.astype(np.float32)
mean_f = raw_features[:-horizon].mean(axis=0)
std_f = raw_features[:-horizon].std(axis=0) + 1e-6
norm_features = (raw_features - mean_f) / std_f

X_seq, y_seq = [], []
for i in range(seq_len, len(norm_features)):
    # Include known future calendar/macro covariates of day i at the end of window
    X_seq.append(norm_features[i-seq_len:i])
    y_seq.append(norm_features[i, 0])
X_seq = np.array(X_seq, dtype=np.float32)
y_seq = np.array(y_seq, dtype=np.float32)
exog_future = norm_features[seq_len:, 1:].astype(np.float32)

X_tr, X_te = torch.tensor(X_seq[:-horizon]).to(device), torch.tensor(X_seq[-horizon:]).to(device)
E_tr, E_te = torch.tensor(exog_future[:-horizon]).to(device), torch.tensor(exog_future[-horizon:]).to(device)
y_tr = torch.tensor(y_seq[:-horizon]).unsqueeze(-1).to(device)

train_loader = DataLoader(TensorDataset(X_tr, E_tr, y_tr), batch_size=32, shuffle=True)

class TemporalAttentionBiLSTM(nn.Module):
    def __init__(self, in_dim=4, hid_dim=64, exog_dim=3):
        super().__init__()
        self.lstm = nn.LSTM(in_dim, hid_dim, num_layers=2, batch_first=True, bidirectional=True, dropout=0.1)
        self.attn = nn.Linear(hid_dim * 2, 1)
        self.head = nn.Sequential(
            nn.Linear(hid_dim * 2 + exog_dim, 64),
            nn.GELU(),
            nn.Linear(64, 1)
        )
    def forward(self, x_seq, x_exog):
        out, _ = self.lstm(x_seq)  # [B, T, 2*H]
        weights = torch.softmax(self.attn(out), dim=1)  # Temporal Attention weights
        context = (weights * out).sum(dim=1)
        return self.head(torch.cat([context, x_exog], dim=-1))

class ResidualNBEATSNet(nn.Module):
    def __init__(self, flat_dim=28*4 + 3, width=128):
        super().__init__()
        self.b1 = nn.Sequential(nn.Linear(flat_dim, width), nn.ReLU(), nn.Linear(width, width), nn.ReLU())
        self.backcast1 = nn.Linear(width, flat_dim)
        self.forecast1 = nn.Linear(width, 1)
        self.b2 = nn.Sequential(nn.Linear(flat_dim, width), nn.ReLU(), nn.Linear(width, width), nn.ReLU())
        self.forecast2 = nn.Linear(width, 1)
    def forward(self, x_seq, x_exog):
        x = torch.cat([x_seq.reshape(x_seq.size(0), -1), x_exog], dim=-1)
        h1 = self.b1(x)
        res = x - self.backcast1(h1)
        h2 = self.b2(res)
        return self.forecast1(h1) + self.forecast2(h2)

def train_dl_model(model, epochs=45, lr=2e-3):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.HuberLoss()
    t0 = time.perf_counter()
    model.train()
    for _ in range(epochs):
        for xb, eb, yb in train_loader:
            opt.zero_grad()
            loss = loss_fn(model(xb, eb), yb)
            loss.backward()
            opt.step()
    elapsed = time.perf_counter() - t0
    model.eval()
    with torch.no_grad():
        preds_norm = model(X_te, E_te).squeeze(-1).cpu().numpy()
    return preds_norm * std_f[0] + mean_f[0], elapsed

bilstm_preds, bilstm_time = train_dl_model(TemporalAttentionBiLSTM().to(device))
nbeats_preds, nbeats_time = train_dl_model(ResidualNBEATSNet().to(device))
print(f'Attention-BiLSTM trained in {bilstm_time:.2f}s | Residual N-BEATS trained in {nbeats_time:.2f}s')

In [ ]:
# 4. Leaderboard Evaluation & Hybrid Neural-Tree-Econometric Ensemble
y_true = test_df['demand_millions'].values
hybrid_all = 0.40 * gbr_preds + 0.30 * bilstm_preds + 0.30 * nbeats_preds

def eval_row(name, preds, runtime_s):
    mape = np.mean(np.abs((y_true - preds) / y_true)) * 100.0
    mae = mean_absolute_error(y_true, preds)
    rmse = np.sqrt(mean_squared_error(y_true, preds))
    r2 = r2_score(y_true, preds)
    return {'Architecture': name, 'MAPE (%)': round(mape, 2), 'MAE (M)': round(mae, 2), 'RMSE (M)': round(rmse, 2), 'R2 Score': round(r2, 4), 'Fit Time (s)': round(runtime_s, 2)}

leaderboard = pd.DataFrame([
    eval_row('1. Statsmodels SARIMAX(1,0,1)(1,0,1,7)', sarimax_preds, sarimax_time),
    eval_row('2. PyTorch Temporal Attention Bi-LSTM (GPU)', bilstm_preds, bilstm_time),
    eval_row('3. PyTorch Residual N-BEATS (GPU)', nbeats_preds, nbeats_time),
    eval_row('4. Scikit-Learn GradientBoosting (Lag Features)', gbr_preds, gbr_time),
    eval_row('5. Hybrid Neural + GBR Meta-Ensemble', hybrid_all, bilstm_time + nbeats_time + gbr_time),
]).sort_values('MAPE (%)').reset_index(drop=True)

print('=== 60-DAY OUT-OF-TIME FORECASTING LEADERBOARD ===')
print(leaderboard.to_string(index=False))